# 2. Joint Index Mapping (Expanded Explanation)

Defines the indices of each joint in the robot's arm. These indices map software commands to physical actuators.

---

## 🔢 What Do These Numbers Actually Mean?

```python
class G1JointIndex:
    LeftShoulderPitch = 15
    LeftShoulderRoll = 16
    LeftShoulderYaw = 17
    LeftElbow = 18
    LeftWristRoll = 19
    LeftWristPitch = 20
    LeftWristYaw = 21

    kNotUsedJoint = 29
```

These numbers (**15 → 21**) are **motor IDs** in the G1’s internal actuator network.

---

## 🧠 Conceptual Model

The Unitree G1 is controlled as a **flat array of motors**:

```text
motor_state[0], motor_state[1], ..., motor_state[28]
```

Each index corresponds to:

* One **physical actuator (motor + gearbox)**
* One **joint DOF (degree of freedom)**

So when you write:

```python
msg.motor_state[15].q
```

you are reading:

> “The current position of motor ID 15”

---

## 🦾 Why Do Arms Start at Index 15?

The indices are **not arbitrary** — they reflect the robot’s full-body layout.

A typical ordering (conceptually) is:

```text
0–5     → Left leg
6–11    → Right leg
12–14   → Torso / waist
15–21   → Left arm
22–28   → Right arm
```

So:

| Joint             | Index | Meaning                          |
| ----------------- | ----- | -------------------------------- |
| LeftShoulderPitch | 15    | First actuator of left arm chain |
| LeftShoulderRoll  | 16    | Second shoulder DOF              |
| LeftShoulderYaw   | 17    | Third shoulder DOF               |
| LeftElbow         | 18    | Elbow flexion                    |
| LeftWristRoll     | 19    | Wrist rotation                   |
| LeftWristPitch    | 20    | Wrist up/down                    |
| LeftWristYaw      | 21    | Wrist twist                      |

---

## 🧩 Why This Matters in Code

When you loop:

```python
for j in self.joints:
    q = msg.motor_state[j].q
```

You are:

* Extracting **joint angles in kinematic order**
* But accessing them via **hardware index order**

---

## ⚠️ Critical Insight

These indices are **hardware-defined**, not algorithm-defined.

That means:

* You **cannot change them**
* You must **match them exactly to the robot firmware**
* A wrong index = commanding the wrong joint

Example mistake:

```python
# WRONG: off-by-one
LeftElbow = 17   # actually controls shoulder yaw
```

→ This leads to:

* Unexpected motion
* Potential instability

---

## 🔄 Relationship to Kinematics

Even though indices are flat, the robot is **hierarchical**:

```text
Shoulder → Elbow → Wrist
```

But the SDK exposes:

```text
motor_state[index]
```

So YOU must impose structure:

```python
self.joints = [
    ShoulderPitch,
    ShoulderRoll,
    ShoulderYaw,
    Elbow,
    WristRoll,
    WristPitch,
    WristYaw
]
```

This creates a **logical kinematic chain on top of a flat hardware array**.

---

## 🧠 Why 7 DOF?

The arm has **7 degrees of freedom**, matching human arm redundancy:

* 3 DOF → shoulder
* 1 DOF → elbow
* 3 DOF → wrist

This allows:

* Multiple ways to reach same position
* Better manipulation flexibility
* RL-friendly redundancy

---

## ⚙️ What is `kNotUsedJoint = 29`?

```python
kNotUsedJoint = 29
```

This is a **special control slot**, not a real joint.

Used for:

```python
self.low_cmd.motor_cmd[29].q = enable_value
```

It acts as:

> ✅ **SDK enable / disable flag**

---

## 🔥 Engineering Insight

From a systems perspective:

* Indices = **network addresses on RS-485 bus**
* Each motor has a **unique ID (0–28)**
* Commands are sent as:

  ```text
  motor_cmd[index] → motor[index]
  ```

This is effectively:

> A **distributed control system over a motor bus**

---

## 🧠 Teaching Insight

This section is a great place to highlight:

> “Robotics software is often a mapping between abstract models and physical hardware addresses.”

Students should understand:

* Code uses **lists and indices**
* Robot uses **motors and wires**
* This mapping is the bridge between them

